#### Imports

In [94]:
import numpy as np
import cupy as cp
import matplotlib.pyplot as plt
import struct
import math

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from tqdm import tqdm
import time
import random

#### Useful functions

In [172]:
### Takes a box and splits it into cubes (blocks) of a specified dimension. Returns
### a list of numpy arrays, each representing one block of the volume. If block_dim
### does not divide equally into the volume dimensions, creates blocks that extend past the
### dimensions of the volume, and populates the coordinates outside the volume with a chosen 
### junk, which is an input to the function
def to_blocks(box, block_dim, junk):
    
    xdim = box.shape[0]
    ydim = box.shape[1]
    zdim = box.shape[2]

    x_steps = int(xdim // block_dim)
    if xdim % block_dim != 0:
        x_steps += 1
    if xdim < block_dim:
        x_steps = 1
    y_steps = int(ydim / block_dim)
    if ydim % block_dim != 0:
        y_steps += 1
    if ydim < block_dim:
        y_steps = 1
    z_steps = int(zdim / block_dim)
    if zdim % block_dim != 0:
        z_steps += 1
    if zdim < block_dim:
        z_steps = 1

    blocks = []
    block_locs = []

    for k in range(z_steps):
        for j in range(y_steps):
            for i in range(x_steps):

                x_lo, y_lo, z_lo = int(i * block_dim), int(j * block_dim), int(k * block_dim)
                x_hi, y_hi, z_hi = min(x_lo + block_dim, xdim), min(y_lo + block_dim, ydim), min(z_lo + block_dim, zdim)

                # create a block, fill with junk
                block = np.full((block_dim, block_dim, block_dim), junk)

                # slice the block from the original box
                block[:x_hi - x_lo, :y_hi - y_lo, :z_hi - z_lo] = box[x_lo:x_hi, y_lo:y_hi, z_lo:z_hi]

                block = block.reshape(-1)
                blocks.append(block)
                block_locs.append((x_lo, y_lo, z_lo))

    return blocks, block_locs



### Reads binary files in input_dir between timesteps min_file and max_file,
### and returns them in a list of numpy arrays, one for each chunk of data (box).
### Also returns a list of the tuples of the location and dimension of each
### box. Finally, returns a list of the number of boxes at each time step.
def process_data(input_dir, min_file, max_file, level, component):

    print("Retrieving data from directory", input_dir)
    
    # create output lists
    boxes, locations, dimensions, box_counts = [], [], [], []

    print("Processing data...")
    for t in tqdm(range(min_file, max_file)): # iterate over each time step
        filename = f"{input_dir}{t}-wholeNewFormat-{component}-{level}.raw"

        # Keeps track of how many boxes are at each time step
        box_count = 0
        boxes_t, locations_t, dimensions_t = [], [], []
        
        with open(filename, "rb") as file:
            while True: # Iterate through all the data at the current timestep until none is left
                test = file.read(4)
                if len(test) < 4: # Break the loop if all data has been read
                    break
                # Read the location of the box
                x = int(struct.unpack('<f', test)[0])
                y = int(struct.unpack('<f', file.read(4))[0])
                z = int(struct.unpack('<f', file.read(4))[0])
                locations_t.append((x, y, z))

                # Read the dimensions of the box
                xdim = int(struct.unpack('<f', file.read(4))[0])
                ydim = int(struct.unpack('<f', file.read(4))[0])
                zdim = int(struct.unpack('<f', file.read(4))[0])
                dimensions_t.append((xdim, ydim, zdim))

                # Read the data in the box
                box = np.empty((xdim, ydim, zdim))
                for k in range(zdim):
                    for j in range(ydim):
                        for i in range(xdim):
                            box[i][j][k] = struct.unpack('<f', file.read(4))[0]
                boxes_t.append(box)
                box_count += 1

        boxes.append(boxes_t)
        locations.append(locations_t)
        dimensions.append(dimensions_t)
        box_counts.append(box_count)

    return boxes, locations, dimensions, box_counts



def create_samples(boxes, block_dim, junk):

    print("Creating samples...")
    
    # # Check if block_dim is a power of 2 (necessary for convolutional autoencoder)
    # if not math.log2(block_dim).is_integer():
    #     print("Invalid block_dim! Dimension must be a power of 2.")
    #     return
    
    samples, block_locs = [], []

    for t in tqdm(range(len(boxes))):
        boxes_t = boxes[t]
        samples_t, block_locs_t = [], []
    
        for box in boxes_t: # Iterate through each box (now we don't care which time step
                                      # it's associated with
            blocks, locs = to_blocks(box, block_dim, junk)
            samples_t.append(blocks)
            block_locs_t.append(locs)

        samples.append(samples_t)
        block_locs.append(block_locs_t)

    return samples, block_locs



def initialize_codewords_random(blocks, num_codewords):
    random_indices = random.sample(range(len(blocks)), num_codewords)
    codewords = []
    for i in random_indices:
        codewords.append(i)
    return codewords


def initialize_codewords_enhanced(blocks, num_codewords, block_dim):

    # Sort by distance to origin
    origin = cp.zeros((block_dim**3), dtype=cp.float32)
    distances = cp.linalg.norm(blocks - origin, axis=1)

    # Sort by sum of values
    sums = cp.sum(blocks, axis=1)

    # Sort indices based on distances and sums
    sorted_d1 = cp.argsort(distances)
    sorted_d2 = cp.argsort(sums)

    # Find initial codewords
    # Split sorted indices into equal parts
    d1_subs = cp.array_split(sorted_d1, num_codewords)
    d2_subs = cp.array_split(sorted_d2, num_codewords)

    codeword_inds = []
    for i in range(num_codewords):
        d1_sub = d1_subs[i]
        d2_sub = d2_subs[i]

        intersection = cp.intersect1d(d1_sub, d2_sub)

        if intersection.size == 0:
            codeword_idx = cp.median(d1_sub)
        else:
            codeword_idx = cp.median(intersection)

        codeword_inds.append(int(codeword_idx))

    codewords = blocks[codeword_inds]
    
    # # Sorting by distance to origin
    # d1 = blocks
    # distances = []
    
    # for block in d1:
    #     block -= block.min()
    #     origin = cp.zeros((block_dim**3))
    #     dist_to_origin = cp.linalg.norm(block - origin)
    #     distances.append(dist_to_origin)

    # d1_distances = zip(distances, range(len(d1)))
    # sorted_d1 = cp.array([idx for _, idx in sorted(d1_distances)])

    # # Sorting by sum of values
    # d2 = blocks
    # sums = []

    # for block in d2:
    #     total = cp.sum(block)
    #     sums.append(total)

    # d2_sums = zip(sums, range(len(d2)))
    # sorted_d2 = cp.array([idx for _, idx in sorted(d2_sums)])


    # # Find initial codewords
    # d1_subs = cp.array_split(sorted_d1, num_codewords)
    # d2_subs = cp.array_split(sorted_d2, num_codewords)

    # codeword_inds = []
    # for i in range(num_codewords):

    #     d1_sub = d1_subs[i]
    #     d2_sub = d2_subs[i]

    #     intersection = cp.intersect1d(d1_sub, d2_sub)
    #     if intersection.size == 0:
    #         codeword_idx = cp.median(d1_sub)
    #     else:
    #         codeword_idx = cp.median(intersection)
    #     codeword_inds.append(int(codeword_idx))

    # codewords = [blocks[k] for k in codeword_inds]

    return codewords


def assign_to_codeword(blocks, codewords, batch_size=1000): 

    # blocks = cp.array(blocks, dtype=cp.float32) # Shape: (num_blocks, block_dim**3)
    # codewords = cp.array(codewords, dtype=cp.float32) # Shape: (num_codewords, block_dim**3)

    num_blocks = blocks.shape[0]
    num_codewords = codewords.shape[0]
    assignments = cp.zeros(num_blocks, dtype=cp.int32)

    for i in range(0, num_blocks, batch_size):
        # Take a batch of blocks
        blocks_batch = blocks[i:i+batch_size]

        # Reshape blocks and codewords for use in linalg.norm
        blocks_reshaped = blocks_batch[:, None, :]
        codewords_reshaped = codewords[None, :, :]

        # Compute euclidean distance between blocks and codewords
        distances = cp.linalg.norm(blocks_reshaped - codewords_reshaped, axis=2)

        # Assign each block to the closest codeword
        assignments[i:i+batch_size] = cp.argmin(distances, axis=1)
    
    # # Reshape blocks and codewords for use in linalg.norm
    # blocks_reshaped = blocks[:, None, :]
    # codewords_reshaped = codewords[None, :, :]

    # # Compute euclidean distance between blocks and codewords
    # distances = cp.linalg.norm(blocks_reshaped - codewords_reshaped, axis=2)
    
    # # Assign each block to the closest codeword
    # assignments = cp.argmin(distances, axis=1)

    return assignments



def update_codewords(blocks, assignments, num_codewords, old_codewords, block_dim):

    # blocks = np.array(blocks)
    # assignments = np.array(assignments)

    new_codewords = cp.zeros((num_codewords, block_dim**3))
    counts = cp.zeros(num_codewords)

    # Accumulate blocks into assigned codewords
    cp.add.at(new_codewords, assignments, blocks)
    cp.add.at(counts, assignments, 1)

    # Avg the codewords
    nonzero_counts = counts > 0
    new_codewords[nonzero_counts] /= counts[nonzero_counts][:, None]
    new_codewords[~nonzero_counts] = old_codewords[~nonzero_counts]
    
    # new_codewords = np.zeros((num_codewords, block_dim**3))
    # counts = np.zeros(num_codewords)

    # for i, block in enumerate(blocks):
    #     codeword_idx = assignments[i]
    #     new_codewords[codeword_idx] += block
    #     counts[codeword_idx] += 1

    # # Average the blocks for each centroid
    # for i in range(num_codewords):
    #     if counts[i] > 0:
    #         new_codewords[i] /= counts[i]
    #     else:
    #         new_codewords[i] = old_codewords[i]

    return new_codewords


def gla(max_iter, blocks, codebook_size):

    all_blocks_list = [block for blocks_t in blocks for sub_blocks in blocks_t for block in sub_blocks]
    all_blocks = cp.empty((len(all_blocks_list), block_dim**3), dtype=cp.float32)
    all_blocks[:] = cp.array(all_blocks_list, dtype=cp.float32)

    codewords = cp.array(initialize_codewords_enhanced(all_blocks, codebook_size, block_dim))
    print("Initialized codewords. Beginning VQ.")
    prev_codewords = cp.array(codewords)

    for iteration in tqdm(range(max_iter)):
        assignments = assign_to_codeword(all_blocks, codewords)
        codewords = update_codewords(all_blocks, assignments, codebook_size, prev_codewords, block_dim)

        delta = cp.sum(cp.linalg.norm(codewords - prev_codewords, axis=1))
        if delta < tol:
            print(f"Converged after {iteration + 1} iterations.")
            break

        prev_codewords = codewords

    final_assignments = []
    all_blocks_idx = 0
    for t in range(len(blocks)):
        assignments_t = []
        blocks_t = blocks[t]
        for box_idx in range(len(blocks_t)):
            sub_assignments = []
            sub_blocks = blocks_t[box_idx]
            for block_idx in range(len(sub_blocks)):
                sub_assignments.append(assignments[all_blocks_idx])
                all_blocks_idx += 1
            assignments_t.append(sub_assignments)
        final_assignments.append(assignments_t)
    
    codebook = (final_assignments, codewords)

    return codebook


def find_block(locs, block_dim, x, y, z):

    block_idx = 0
    rem = (0, 0, 0)
    for l in range(len(locs)):
        loc = locs[l]
        if x >= loc[0] and x < loc[0] + block_dim:
            if y >= loc[1] and y < loc[1] + block_dim:
                if z >= loc[2] and z < loc[2] + block_dim:
                    rem = (x - loc[0], y - loc[1], z - loc[2])
                    block_idx = l
                    break
    # for i in range(3):
    #     if rem[i] >= block_dim:
    #         print((x, y, z), loc, rem)
    return block_idx, rem

    

def reconstruct_box(dim, block_locs, block_dim, codewords, assignments):

    reconstructed = np.zeros((dim[0], dim[1], dim[2]))
    
    for z in range(dim[2]):
        for y in range(dim[1]):
            for x in range(dim[0]):
                
                block_idx, rem = find_block(block_locs, block_dim, x, y, z)
                # if rem[0] > 31:
                #     print(x, y, z)
                #     print(rem)
                #     print(block_idx)
                codebook_idx = assignments[block_idx]
                codeword = codewords[int(codebook_idx)]
                
                codeword = codeword.reshape(block_dim, block_dim, block_dim)
                pix = codeword[rem[0], rem[1], rem[2]]
                reconstructed[x][y][z] = pix

    return reconstructed



def calc_avg_rmse(actuals, regens):

    rmses = []

    for t in range(len(actuals)):
        boxes_t = actuals[t]
        regen_boxes_t = regens[t]
    
        for box_idx in range(len(boxes_t)):

            actual = boxes_t[box_idx]
            pred = regen_boxes_t[box_idx]
        
            xdim = actual.shape[0]
            ydim = actual.shape[1]
            zdim = actual.shape[2]

            sum = 0
        
            for k in range(zdim):
                for j in range(ydim):
                    for i in range(xdim):
                        sq = (actual[i][j][k] - pred[i][j][k])**2
                        sum += sq

            rmse = (sum / (xdim * ydim * zdim))**0.5
            rmses.append(rmse)

    rmses = np.array(rmses)
    return rmses.mean()

#### Hyperparameters

In [161]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on", device)
min_file = 74
max_file = 75
block_dim = 2
level = 0
component = 6
junk = 0
max_iter = 100
tol = 1e-4
codebook_size = 4096
dir = "wholeVolumesNewFormat-" + str(component) + "-" + str(level) + "/"

Running on cuda


#### Data preprocessing

In [151]:
boxes, locations, dimensions, num_boxes = process_data(dir, min_file, max_file, level, component)

blocks, block_locs = create_samples(boxes, block_dim, junk)
# print(len(blocks))
# print(len(blocks[0]))
# print(blocks[0][0].shape)

Retrieving data from directory wholeVolumesNewFormat-6-0/
Processing data...


100%|█████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  1.10it/s]


Creating samples...


100%|█████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  1.20it/s]


#### Generalized Llyod Algorithm to create codebook

In [173]:
codebook = gla(max_iter, blocks, codebook_size)

Initialized codewords. Beginning VQ.


100%|█████████████████████████████████████████████████████████████████████████████████| 100/100 [05:50<00:00,  3.51s/it]


#### Reconstruct timestep from codebook

In [174]:
final_assignments = codebook[0]
codewords = codebook[1]

regen_boxes = []

for t in range(len(blocks)): # iterating over time

    dimensions_t = dimensions[t]
    block_locs_t = block_locs[t]
    assignments_t = final_assignments[t]
    regen_boxes_t = [None] * len(dimensions_t)

    for box_idx in tqdm(range(len(dimensions_t))):
        regen_boxes_t[box_idx] = reconstruct_box(dimensions_t[box_idx], block_locs_t[box_idx], 
                                block_dim, codewords, assignments_t[box_idx])

    regen_boxes.append(regen_boxes_t)

rmse = calc_avg_rmse(boxes, regen_boxes)
print(rmse)

100%|█████████████████████████████████████████████████████████████████████████████████| 576/576 [01:31<00:00,  6.29it/s]


14.218211016296323
